<a href="https://colab.research.google.com/github/ZiadSakr5/ZezoSakr/blob/main/CNN%20section.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch version: 2.9.0+cpu
Device: cpu


In [15]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm

device = xm.xla_device()

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.9.0+cpu
Device: xla:0


/tmp/ipykernel_918/3130186495.py:6: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


In [4]:
# Hyperparameters
BATCH_SIZE   = 32
LEARNING_RATE = 0.001
EPOCHS       = 10
NUM_CLASSES  = 10
IMAGE_SIZE   = 32

# Use GPU if available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {DEVICE}")

Training on: cpu


In [5]:
# Transforms
# Normalize using CIFAR-10 channel means and stds
# Use PIL to convert directly to tensor, bypassing numpy issue with torch.from_numpy
def pil_to_tensor(img):
    """Convert PIL Image to tensor using PyTorch without numpy intermediary."""
    # Convert PIL Image to tensor directly
    img_array = np.array(img, dtype=np.uint8)
    # Convert numpy array to torch tensor using torch.tensor (not from_numpy)
    tensor = torch.tensor(img_array, dtype=torch.float32) / 255.0
    return tensor.permute(2, 0, 1)  # HWC -> CHW

transform = transforms.Compose([
    transforms.Lambda(pil_to_tensor),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std =(0.2470, 0.2435, 0.2616))
])

# Download & Load
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True,  download=True, transform=transform)

test_dataset  = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# CIFAR-10 class names
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")
print(f"Classes          : {CLASS_NAMES}")

100%|██████████| 170M/170M [00:05<00:00, 31.0MB/s]


Training samples : 50,000
Test samples     : 10,000
Classes          : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [6]:
class CNN(nn.Module):

    def __init__(self, num_classes: int = 10):
        super(CNN, self).__init__()

        # Convolutional Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Convolutional Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Convolutional Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Flatten + Fully Connected Layers
        # Spatial size after 3 blocks: floor((32-2)/2-2)/2-2)/2) = 2
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 128),   # 256 → 128
            nn.ReLU(),
            nn.Linear(128, num_classes)    # 128 → 10
        )

    def forward(self, x):
        x = self.block1(x)       # Conv → ReLU → Pool
        x = self.block2(x)       # Conv → ReLU → Pool
        x = self.block3(x)       # Conv → ReLU → Pool
        x = self.classifier(x)   # Flatten → FC → ReLU → FC
        return x


# Instantiate & inspect
model = CNN(num_classes=NUM_CLASSES).to(DEVICE)
print(model)

# Quick parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

CNN(
  (block1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Total trainable parameters: 57,770


In [7]:
# CrossEntropyLoss combines LogSoftmax + NLLLoss (standard for classification)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Loss      : {criterion.__class__.__name__}")
print(f"Optimizer : Adam  (lr={LEARNING_RATE})")

Loss      : CrossEntropyLoss
Optimizer : Adam  (lr=0.001)


In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Run one full pass over the training set."""
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # clear gradients
        outputs = model(images)        # forward pass
        loss    = criterion(outputs, labels)  # compute loss
        loss.backward()                # backpropagation
        optimizer.step()               # update weights

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    return epoch_loss, epoch_acc


print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>9}")
print("-" * 32)

for epoch in range(1, EPOCHS + 1):
    loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    print(f"{epoch:>6}  {loss:>10.4f}  {acc:>8.2f}%")

 Epoch  Train Loss  Train Acc
--------------------------------
     1      1.5227     44.11%
     2      1.1712     58.38%
     3      1.0197     64.23%
     4      0.9231     67.56%
     5      0.8487     70.15%
     6      0.7911     72.31%
     7      0.7441     74.02%
     8      0.7100     75.04%
     9      0.6700     76.51%
    10      0.6367     77.63%


In [18]:
def evaluate(model, loader, device):
    """Compute accuracy on a given DataLoader (no gradient needed)."""
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

    return 100.0 * correct / total


final_accuracy = evaluate(model, test_loader, DEVICE)
print(f"Final Test Accuracy: {final_accuracy:.2f}%")

Final Test Accuracy: 69.62%
